# Regresión de potencia eléctrica: preparación

Consulta la [metodología](../docs/metodologia.md) y ejecuta las etapas en orden. Los comentarios se calculan con los resultados guardados.

In [1]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("power_plant_regression")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre este notebook desde su proyecto independiente.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from power_plant_regression.alerts import configurar_notebook, mostrar_alertas
from IPython.display import display, Markdown, Image
configurar_notebook()


In [2]:
from power_plant_regression.analysis import run_stage
import pandas as pd
metrics,comment=run_stage(PROJECT_ROOT,"preparacion")
display(Markdown(comment))
scalars={k:v for k,v in metrics.items() if not isinstance(v,(dict,list))}
if scalars:display(pd.DataFrame(list(scalars.items()),columns=["Indicador","Valor"]))
if "candidatos" in metrics:display(pd.DataFrame(metrics["candidatos"]))
for filename in ["eda.png","evaluacion.png"]:
    path=PROJECT_ROOT/"reports/preparacion"/filename
    if path.exists():display(Image(filename=str(path)))
for filename in ["estimaciones.csv","metricas_test.csv","horas_por_empleado.csv"]:
    path=PROJECT_ROOT/"reports/preparacion"/filename
    if path.exists():display(pd.read_csv(path).head(20))

# Regresión de potencia eléctrica: preparacion

Se utiliza **solo la primera hoja (Sheet1)**: las otras hojas son reordenamientos del mismo conjunto, no observaciones independientes. Se retiran **41 duplicados exactos** y se agrupan condiciones idénticas de entrada para que no crucen particiones. Separación aproximada por grupos: **60 % train, 10 % validación, 10 % calibración y 20 % test**. Objetivo: potencia eléctrica en **MW**. No se dispone de fecha para validar un pronóstico temporal.


,Indicador,Valor
0,fuente,https://archive.ics.uci.edu/dataset/294/combin...
1,sha256,cc7b2a4977c0a44e8221c91d9a7e5746b3c68186cff7e5...
2,hoja_utilizada,Sheet1
3,filas_hoja,9568
4,duplicados_exactos_retirados,41
5,objetivo_unidad,Potencia eléctrica PE en MW


In [3]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/01_preparacion_datos.json")
from power_plant_regression.informe import generar_informe
generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))

[Abrir informe integral](../reports/informe_final/informe_final.md)